# Man-in-the-Middle — Selective Hijack in a Fleet

Three drones fly straight north from spread-out homes, all monitored by a
**single GCS process** (they share one `SimGCS`). An attacker has man-in-the-
middled **only the middle drone (sysid 2)**: it runs a real `Intervention` —
the same trigger + guided plan a `gcs.intervene(...)` call would build — from
the MITM position. Once the drone reaches a chosen mission point
(`MISSION_CURRENT.seq >= 3`) the MITM drives it to attacker-chosen coordinates,
spoofed to look like the GCS (sysid 255).

The other two drones have **no MITM** — their telemetry and commands reach the
GCS directly. The point of this scenario is that `.mitm` is a per-vehicle
attribute, so one vehicle can be compromised while its fleet-mates on the same
GCS are untouched.

Expected result: the green and red drones fly their full north missions; the
blue drone turns **west** mid-mission with no command from the GCS.

In [ ]:
from simulator import Oracle, Simulator
from simulator.config import DATA_PATH, PARAMS_PATH, Color, Model
from simulator.entities import Intervention, MissionTrigger, SimGCS, SimVehicle
from simulator.helpers import clean
from simulator.helpers.coordinates import ENU, ENUPose, GRAPose
from simulator.planner import AutoPlan, InterventionPlan
from simulator.runtime.mitm import InterventionStrategy
from simulator.visualizer import Gazebo, GazMarker

clean()

## Origin, fleet layout, and waypoints

In [ ]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241, alt=0, heading=0)
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading)

speed = 5.0  # m/s
cruise_alt = 10.0  # m
model = Model.IRIS

# Three drones on one GCS, packed close together so all are visible at once.
sysids = [1, 2, 3]
colors = [Color.GREEN, Color.BLUE, Color.RED]
homes = ENUPose.list(
    [  # east, north, up, heading
        (-10.0, 0.0, 0.0, 0),
        (0.0, 0.0, 0.0, 0),
        (10.0, 0.0, 0.0, 0),
    ]
)

# Each drone flies the same short north mission relative to its own home.
# Mission seq: seq=0 home, seq=1 TAKEOFF, seq=2 flying to north_20,
#              seq=3 flying to north_40 ← attacker hijacks here
home_wp = ENU(x=0, y=0, z=0)
north_20 = ENU(x=0, y=20, z=cruise_alt)
north_40 = ENU(x=0, y=40, z=cruise_alt)
mission_wps = [home_wp, north_20, north_40]

# The attacker compromises only the middle drone.
hijacked_sysid = 2

# Attacker's chosen destination: 30 m WEST of origin (opposite the mission), as
# an ENU relative to the run origin. The InterventionPlan converts it to
# geodetic when it binds inside the MITM.
hijack_target = ENU(x=-30, y=0, z=cruise_alt)
print(f"Hijacking sysid {hijacked_sysid} → ENU {hijack_target}")

## Vehicles (all on one shared GCS)

In [ ]:
# A single shared SimGCS instance puts all three vehicles under one GCS process.
gcs = SimGCS(name=f"FLEET_{''.join([color.emoji for color in colors])}")

mission_folder = DATA_PATH / "missions"

vehs: list[SimVehicle] = []
for sysid, color, home in zip(sysids, colors, homes, strict=True):
    mission_path = str(mission_folder / f"mitm_fleet_{sysid}.waypoints")
    plan = AutoPlan.from_relative_path(
        name="north_mission",
        sysid=sysid,
        gra_origin=gra_origin,
        relative_home=home,
        relative_path=mission_wps,
        mission_path=mission_path,
        navigation_speed=speed,
        firmware=model.firmware,
    )

    veh = SimVehicle.from_relative(
        sysid=sysid,
        gcss=[gcs],
        plan=plan,
        color=color,
        enu_origin=enu_origin,
        relative_home=home,
        relative_path=mission_wps,
        model=model,
    )
    vehs.append(veh)


## Visualizer

In [ ]:
gaz = Gazebo(gra_origin, world_path="simulator/visualizer/gazebo/worlds/runway.world")
origin_marker = GazMarker(
    name="origin",
    group="origin",
    pos=enu_origin.unpose(),
    color=Color.WHITE,
)
gaz.markers.append(origin_marker)


## Simulator + selective MITM hijack

In [ ]:
orac = Oracle()

for veh in vehs:
    orac.add_vehicle(veh)

# No GCS intervention here — the attacker does the redirection. `.mitm` is a
# per-vehicle attribute, so the MITM is interposed on ONLY the hijacked drone.
# The other two share the same GCS but talk to it directly.
hijacked_veh = next(v for v in vehs if v.sysid == hijacked_sysid)
hijacked_veh.mitm = InterventionStrategy(
    Intervention(
        trigger=MissionTrigger(seq=3),
        plan=InterventionPlan.from_relative_path(
            relative_path=[hijack_target],
            enu_origin=enu_origin,
            firmware=model.firmware,
            land=False,
        ),
        firmware=model.firmware,
    )
)

simulator = Simulator(oracle=orac, visualizer=gaz, verbose=1)

simulator.preview()

In [ ]:
simulator.run()


## What to observe

- **Green (sysid 1)** and **red (sysid 3)** fly their full north missions.
- **Blue (sysid 2)** turns **west** mid-mission — driven entirely by the man-in-
  the-middle, with no command from the GCS.
- `simulator/logs/mitm/` — only `mitm_2.log` exists (`MITM intervention: taking
  over vehicle 2`). There is no `mitm_1.log` / `mitm_3.log`; those drones have
  no proxy.
- `simulator/logs/logics/logic_2.log` — `GCS→SITL forwarding SET_MODE` /
  `COMMAND_INT` (Logic forwards the spoofed commands to the flight controller).
- `simulator/logs/GCSs/GCS_FLEET_*.log` — a single GCS process monitoring all
  three vehicles, with **no** `GCS intervention` line (the redirect did not
  originate from the GCS).